<a href="https://colab.research.google.com/github/SyedHarshath/GenerativeAI/blob/main/Text_To_Video_Generator.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from diffusers import DiffusionPipeline
import torch
import numpy as np
import imageio
from PIL import Image

pipe = DiffusionPipeline.from_pretrained(
    "damo-vilab/text-to-video-ms-1.7b",
    torch_dtype=torch.float16,
    variant="fp16"
).to("cuda")

prompt = "cat doing a somersault"
duration = 10
fps = 8
total_frames = duration * fps

video_batches = pipe(prompt, num_inference_steps=50).frames

target_resolution = (512, 512)

processed_frames = []

for i, batch in enumerate(video_batches):
    for j, frame in enumerate(batch):

        if frame.dtype != np.uint8:
            frame = (frame * 255).clip(0, 255).astype(np.uint8)

        if frame.ndim == 2:
            frame = np.stack([frame] * 3, axis=-1)
        elif frame.ndim == 3 and frame.shape[2] == 1:
            frame = np.repeat(frame, 3, axis=2)
        elif frame.ndim == 3 and frame.shape[2] > 4:
            frame = frame[:, :, :3]

        image = Image.fromarray(frame)
        image = image.resize(target_resolution, Image.LANCZOS)
        processed_frames.append(np.array(image))

        if len(processed_frames) >= total_frames:
            break
    if len(processed_frames) >= total_frames:
        break

if len(processed_frames) < total_frames:
    processed_frames = processed_frames * (total_frames // len(processed_frames)) + processed_frames[:total_frames % len(processed_frames)]

print(f"✅ Total processed frames: {len(processed_frames)}")

output_path = "high_quality_video_10sec.mp4"
writer = imageio.get_writer(
    output_path,
    fps=fps,
    codec='libx264',
    bitrate="5M",
    quality=10
)

for frame in processed_frames:
    writer.append_data(frame)

writer.close()

print(f"🎥 High-quality 10-second video saved as {output_path}")



/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model_index.json:   0%|          | 0.00/384 [00:00<?, ?B/s]

Fetching 12 files:   0%|          | 0/12 [00:00<?, ?it/s]

config.json:   0%|          | 0.00/644 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/460 [00:00<?, ?B/s]

...

diffusion_pytorch_model.fp16.safetensors:   0%|          | 0.00/167M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/657 [00:00<?, ?B/s]

Loading pipeline components...:   0%|          | 0/5 [00:00<?, ?it/s]

  0%|          | 0/50 [00:00<?, ?it/s]

✅ Total processed frames: 80
🎥 High-quality 10-second video saved as high_quality_video_10sec.mp4
